# Week 2, Tuesday: Decision Trees & k-Nearest Neighbors
Datasets: **Wine** (3 wine types, 13 chemical features) and **Breast Cancer** (malignant vs benign, 30 features). Both come built into scikit-learn.

Run top to bottom. Cells marked 🧪 are exercises for you. Write your answers in your own words in the Markdown cells below them.

## 0. Setup

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
sns.set_theme(style='whitegrid')
np.random.seed(42)

## 1. Load and explore both datasets

In [ ]:
wine = load_wine(as_frame=True)
cancer = load_breast_cancer(as_frame=True)
for name, d in [('Wine', wine), ('Breast Cancer', cancer)]:
    print(name, '->', d.data.shape, '| classes:', dict(zip(d.target_names, np.bincount(d.target))))
display(wine.data.head())

In [ ]:
# Do the features live on different scales? (matters a lot for kNN)
summary = wine.data.describe().loc[['min','max']].T
summary['range'] = summary['max'] - summary['min']
summary.sort_values('range', ascending=False).round(2)

🧪 **Exercise 1:** Which Wine feature has the largest range and which the smallest? Predict what this will do to kNN before we test it.

*Your answer:* 

## 2. Decision Tree: how it works
A tree asks a series of yes/no questions about the features (for example *proline <= 755?*) and each question splits the data into purer groups. It chooses the split that reduces **impurity** the most. **Gini impurity** is 0 when a group contains only one class, and higher when classes are mixed.

In [ ]:
Xw_tr, Xw_te, yw_tr, yw_te = train_test_split(wine.data, wine.target, test_size=0.25, random_state=42, stratify=wine.target)
tree = DecisionTreeClassifier(random_state=42).fit(Xw_tr, yw_tr)
print('Default tree  train:', round(tree.score(Xw_tr, yw_tr), 3), ' test:', round(tree.score(Xw_te, yw_te), 3))
print('Tree depth:', tree.get_depth(), ' leaves:', tree.get_n_leaves())

In [ ]:
# Plot the top of the tree (first 2 levels) so it is readable
small = DecisionTreeClassifier(max_depth=2, random_state=42).fit(Xw_tr, yw_tr)
plt.figure(figsize=(14, 6))
plot_tree(small, feature_names=wine.feature_names, class_names=list(wine.target_names), filled=True, rounded=True, fontsize=10)
plt.show()

**How to read the tree:** each box shows the question, the **gini** impurity, the number of **samples**, how many are in each class (**value**) and the majority **class**. Follow *True* to the left and *False* to the right.

🧪 **Exercise 2:** What is the first question (the root split)? Why do you think the tree picked that feature first? What is the gini value in the root and in its children?

*Your answer:* 

In [ ]:
imp = pd.Series(tree.feature_importances_, index=wine.feature_names).sort_values()
imp.tail(8).plot.barh(title='Top feature importances (Decision Tree, Wine)'); plt.show()

## 3. Overfitting in trees: vary `max_depth`
A very deep tree can keep splitting until it memorises the training data. Watch train vs test accuracy as depth grows.

In [ ]:
depths = range(1, 16)
tr, te = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(Xw_tr, yw_tr)
    tr.append(m.score(Xw_tr, yw_tr)); te.append(m.score(Xw_te, yw_te))
plt.plot(depths, tr, 'o-', label='train'); plt.plot(depths, te, 'o-', label='test')
plt.xlabel('max_depth'); plt.ylabel('accuracy'); plt.title('Decision Tree on Wine'); plt.legend(); plt.show()

In [ ]:
# A single test split is noisy, so confirm with 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_means = [cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=42), wine.data, wine.target, cv=cv).mean() for d in depths]
plt.plot(depths, cv_means, 'o-'); plt.xlabel('max_depth'); plt.ylabel('CV accuracy'); plt.title('5-fold CV score vs depth (Wine)'); plt.show()
print('Best depth by CV:', list(depths)[int(np.argmax(cv_means))], ' score:', round(max(cv_means), 3))

🧪 **Exercise 3:** At what depth does the **training** accuracy reach 1.0? Does the test/CV accuracy keep improving after that? Explain what happens when `max_depth` is huge. Then try `min_samples_leaf=5` on a deep tree: does it help?

*Your answer:* 

## 4. k-Nearest Neighbors: how it works
To classify a new point, kNN finds the **k closest training points** (by distance) and takes a majority vote. There is no real training step: it just stores the data.

Because it uses **distance**, features with big numbers dominate. So scaling should matter a lot.

In [ ]:
knn_raw = KNeighborsClassifier(n_neighbors=5).fit(Xw_tr, yw_tr)
knn_sc = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(Xw_tr, yw_tr)
print('kNN WITHOUT scaling  test accuracy:', round(knn_raw.score(Xw_te, yw_te), 3))
print('kNN WITH scaling     test accuracy:', round(knn_sc.score(Xw_te, yw_te), 3))

# More reliable: 5-fold CV (the scaler is refit inside each fold by the pipeline)
raw_cv = cross_val_score(KNeighborsClassifier(n_neighbors=5), wine.data, wine.target, cv=cv)
sc_cv = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)), wine.data, wine.target, cv=cv)
print(f'CV without scaling: {raw_cv.mean():.3f}   CV with scaling: {sc_cv.mean():.3f}')

🧪 **Exercise 4:** How big is the gap? Using the feature ranges from Exercise 1, explain in your own words why scaling changes kNN's results so much.

*Your answer:* 

## 5. Choosing k: small k overfits, large k underfits

In [ ]:
ks = range(1, 31)
tr, te, cvm = [], [], []
for k in ks:
    m = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=k))
    m.fit(Xw_tr, yw_tr)
    tr.append(m.score(Xw_tr, yw_tr)); te.append(m.score(Xw_te, yw_te))
    cvm.append(cross_val_score(m, wine.data, wine.target, cv=cv).mean())
plt.plot(ks, tr, 'o-', label='train'); plt.plot(ks, te, 'o-', label='test'); plt.plot(ks, cvm, 'o-', label='5-fold CV')
plt.xlabel('k (number of neighbours)'); plt.ylabel('accuracy'); plt.title('kNN on Wine (scaled)'); plt.legend(); plt.show()
print('k=1  train:', round(tr[0], 3), ' test:', round(te[0], 3))
print('Best k by CV:', list(ks)[int(np.argmax(cvm))], ' score:', round(max(cvm), 3))

🧪 **Exercise 5:** What happens to the training accuracy when k=1? Why is it exactly that value (think about which point is the nearest neighbour of a training point)? What happens to the curves for large k?

*Your answer:* 

## 6. Repeat on Breast Cancer: Tree vs kNN (5-fold CV)

In [ ]:
Xc, yc = cancer.data, cancer.target
models = {
    'Decision Tree (depth 3)': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Decision Tree (no limit)': DecisionTreeClassifier(random_state=42),
    'kNN k=5 (no scaling)': KNeighborsClassifier(n_neighbors=5),
    'kNN k=5 (scaled)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}
res = {n: cross_val_score(m, Xc, yc, cv=cv) for n, m in models.items()}
table = pd.DataFrame({n: [s.mean(), s.std()] for n, s in res.items()}, index=['mean accuracy', 'std']).T.round(3)
display(table)
table['mean accuracy'].plot.barh(xlim=(0.85, 1.0), title='Breast Cancer: 5-fold CV accuracy'); plt.show()

🧪 **Exercise 6:** Which model won on Breast Cancer? Is the difference big or small compared with the standard deviation? Did scaling help kNN as much as it did on Wine? (Hint: look at the feature ranges in Breast Cancer: `cancer.data.describe()`.)

*Your answer:* 

## 7. Summary notes (fill in, in your own words)
**How a decision tree chooses a split (Gini impurity):** 

**Why deep trees overfit, and two ways to prevent it:** 

**How kNN makes a prediction:** 

**Why kNN needs scaling but trees don't:** 

**What happens when k=1, and when k is very large:** 
